# 05 — Treinar e selecionar CNNs usando validation

## Objetivo
Executar um plano pré-declarado, selecionar checkpoints por validation e comparar receitas sobre as mesmas seeds. Nesta migração, treinamento real fica desativado para revisão.

## Entradas
Certificado v2; train/validation; critério derivado de baseline v2 e registrado antes do plano.

## Saídas
`models/experimental_v2/runs/<run_id>/` e `reports/experimental_v2/selections/<selection_id>/`.

## Permissões de dados
Somente train/validation. Não carregar calibration, holdout, final test, scores v1 ou checkpoints ajustados ao split antigo.

## Reprodutibilidade
Seeds 42/43/44. Configuração explícita, recipe_id por conteúdo, commit/dirty flag, snapshot do código e versões do ambiente por run.

Protocolo: [experimental_protocol_v2.md](../docs/experimental_protocol_v2.md).
Os resultados anteriores estão catalogados como `legacy_v1`; não comparar seus scores diretamente aos v2.


In [ ]:
from pathlib import Path
import sys

# Bootstrap de importação; a resolução dos dados fica nos módulos compartilhados.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "scripts/experimental_protocol.py").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from scripts.v2_artifacts import repository_root, read_json, run_id
ROOT = repository_root()

from scripts.v2_data import load_development, experiment_identity
from scripts.v2_training import (default_config, resolve_config, recipe_identity, register_plan,
                                  read_plan, train_run, select_candidate)
from scripts.v2_artifacts import relative_path
EXECUTE_TRAINING = False
EXECUTE_SELECTION = False
CRITERIA_PATH = None  # caminho relativo do critério registrado no NB03
EXISTING_PLAN_PATH = None  # explícito; nunca escolher automaticamente o mais recente
EXPERIMENT_ID = run_id("cnn_v2")
TRAINING_SEEDS = [42, 43, 44]
TRAIN_CONFIG = default_config("resnet50", seed=42)
# A segunda arquitetura só entra se estiver declarada ANTES dos runs.
RECIPES = [TRAIN_CONFIG]
frames, certificate = load_development(ROOT)
display(experiment_identity(certificate))
display(TRAIN_CONFIG)


## Configuração e escolhas preservadas
Mantemos ResNet50, pesos ImageNet identificados, resolução 224, AdamW, cosine scheduler, cross entropy com label smoothing e pesos calculados exclusivamente em train.
O backbone inteiro é ajustado (`freeze_backbone_epochs=0`). Sampling ponderado e pesos de classe não podem ser combinados.

Flip, rotação e color jitter ficam na configuração salva. Avaliação e inferência compartilham EXIF → RGB → resize bilinear → normalização.
`resolve_config()` registra device e AMP efetivamente utilizados; não salva `auto` como se fosse hardware real.

Downloads de pesos ficam desativados por padrão. Se os pesos genéricos não estiverem em cache, preparar explicitamente sua aquisição antes do treino; não substituir por pesos v1.


In [ ]:
effective_recipes = [resolve_config(c) for c in RECIPES]
display(pd.DataFrame([{ "recipe_id": recipe_identity(c), "architecture": c["architecture"],
                        "device": c["device"], "amp": c["amp"], "epochs": c["epochs"],
                        "batch_size": c["batch_size"], "seeds": TRAINING_SEEDS} for c in effective_recipes]))
plan_path = relative_path(ROOT, EXISTING_PLAN_PATH) if EXISTING_PLAN_PATH else None
if EXECUTE_TRAINING and plan_path is None:
    if CRITERIA_PATH is None:
        raise ValueError("Registre os critérios do NB03 antes de iniciar CNNs")
    plan_path = register_plan(ROOT, EXPERIMENT_ID, effective_recipes, TRAINING_SEEDS, relative_path(ROOT, CRITERIA_PATH))
if plan_path is not None:
    display(read_plan(plan_path))
else:
    print("Prévia de configuração. Nenhum plano ou treinamento CNN foi iniciado.")


## Execução por receita e seed
Cada tentativa recebe run_id, timestamp e provenance. O plano exige uma tentativa por receita/seed; uma falha permanece registrada.
Não há deduplicação por “última execução”. Um novo ciclo após falha requer novo plano e explicação, sem agregar réplicas como evidências independentes.

Durante cada época: registrar train loss, validation loss, accuracy, macro-F1 e learning rate.
As losses usam a mesma CE ponderada/suavizada, média por imagem; validation NLL sem pesos também é registrada para interpretar probabilidades.

**Checkpoint:** maior macro-F1 de validation, empate mantém época mais antiga. **Early stopping:** três épocas sem melhora, conforme config. NaN/Inf invalida o run.


In [ ]:
run_directories = []
if EXECUTE_TRAINING:
    plan = read_plan(plan_path)
    for recipe_id in plan["recipes"]:
        for seed in plan["seeds"]:
            directory = train_run(ROOT, plan_path, recipe_id, seed)
            run_directories.append(directory)
            display(read_json(directory / "run_summary.json"))
else:
    print("Treinamento desativado para revisão da estrutura.")


## Curvas e artefatos por execução
Os logits correspondem à ordem dos IDs no CSV de validation e ao hash do checkpoint. O manifesto permite recalcular métricas sem executar a CNN.
O checkpoint destina-se à inferência; retomada exata de treino não foi implementada nesta fase.


In [ ]:
for directory in run_directories:
    history = pd.read_csv(directory / "training_history.csv")
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    history.plot(x="epoch", y=["train_loss", "validation_loss"], ax=axes[0])
    axes[0].set(title="Loss por época — mesma função de custo", xlabel="Época", ylabel="CE ponderada/suavizada por imagem")
    history.plot(x="epoch", y="validation_macro_f1", ax=axes[1])
    axes[1].set(title="Checkpoint selecionado por validation", xlabel="Época", ylabel="Macro-F1")
    fig.suptitle(directory.name); fig.tight_layout(); plt.show()
    display(read_json(directory / "validation_interval.json"))


## Comparar seeds e selecionar candidato
A seleção exige todas as combinações receita × seed previstas, completas e íntegras.
Comparamos média de macro-F1, desvio-padrão amostral, mínimo, máximo e accuracy. Empate entre receitas usa accuracy média e depois recipe_id.
Dentro da receita selecionada: macro-F1, accuracy e menor seed. A melhor seed não representa a média da receita.

Tempo, parâmetros e tamanho do checkpoint contextualizam os custos. Latência não medida é registrada como ausente; não se presume comparabilidade entre hardwares.
O intervalo bootstrap por lesão é de validation, condicionado à seleção — não uma avaliação final independente.


In [ ]:
selection = None
if EXECUTE_SELECTION:
    if plan_path is None:
        raise ValueError("Selecione explicitamente o plano com todos os runs completos")
    selection_dir = ROOT / "reports/experimental_v2/selections" / run_id("selection")
    selection = select_candidate(ROOT, plan_path, selection_dir)
    runs = pd.read_csv(selection_dir / "runs.csv")
    display(runs[["architecture", "recipe_id", "seed", "best_epoch", "validation_macro_f1", "validation_accuracy", "stop_reason"]])
    display(pd.read_csv(selection_dir / "recipes.csv"))
    display(selection)
    print("Entrada explícita para NB06:", selection_dir.relative_to(ROOT).as_posix())
else:
    print("Seleção pendente; nenhum candidato v1 é usado como substituto.")


## Conclusão
Uma receita só pode ser comparada após completar as seeds declaradas. Escolhas usam exclusivamente validation. O resultado continua sendo evidência interna de desenvolvimento, com limitações de paciente e exposição histórica.

## Artefatos produzidos
Config, environment, provenance/snapshot, plan, pesos de classe, histórico, checkpoint, métricas/logits/IDs de validation, intervalo e manifesto de hashes. Seleção em diretório próprio.

## Próxima etapa
NB06 recebe explicitamente a seleção persistida, verifica o pacote e congela o candidato. Nenhuma política de confiança foi criada.
